# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omi290/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!pip -q install duckdb huggingface_hub pyarrow

In [2]:
from google.colab import userdata
from huggingface_hub import hf_hub_download
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

In [3]:
march_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    token=HF_TOKEN,
)

print(march_path)

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet


In [4]:
db = duckdb.connect()

db.sql(f"""
CREATE OR REPLACE VIEW march_data AS
SELECT *
FROM read_parquet('{march_path}')
""")

In [5]:
db.sql("""
SELECT *
FROM march_data
LIMIT 5
""").df()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

# Feature Vector

The feature vector is built using historical search and engagement metrics that are available before the prediction date. Missing values are handled conservatively, and no future information or label-derived fields are included.

In [6]:
feature_vector = db.sql("""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,

    COALESCE(gsc_impressions,0) AS gsc_impressions,
    COALESCE(gsc_clicks,0) AS gsc_clicks,
    COALESCE(gsc_avg_position,0) AS gsc_avg_position,
    COALESCE(ga4_pageviews,0) AS ga4_pageviews,
    COALESCE(ga4_sessions,0) AS ga4_sessions,
    COALESCE(scroll_events,0) AS scroll_events,

    CASE
        WHEN gsc_impressions < 100 THEN 'Low'
        WHEN gsc_impressions < 1000 THEN 'Medium'
        ELSE 'High'
    END AS impression_bucket

FROM march_data
WHERE ga4_data_available IS TRUE
""").df()

feature_vector.head()

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_sessions,scroll_events,impression_bucket
0,2026-03-01,client_65de48885f4ef01b,content_09be8cc7fcb222af,0,0,0.0,1,1,0,Low
1,2026-03-01,client_65de48885f4ef01b,content_851afac9fe13612e,0,0,0.0,1,1,0,Low
2,2026-03-01,client_65de48885f4ef01b,content_cee6c6fc8c51af14,0,0,0.0,1,1,0,Low
3,2026-03-01,client_65de48885f4ef01b,content_5e120e972f11f833,0,0,0.0,1,1,0,Low
4,2026-03-01,client_65de48885f4ef01b,content_16a7291bb6ecaebe,0,0,0.0,1,1,0,Low


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

# Feature Notes

| Feature | Meaning | Missing Values | Categorical | Available When? |
|---------|---------|----------------|-------------|-----------------|
| gsc_impressions | Search visibility | Filled with 0 | No | Before prediction |
| gsc_clicks | Organic clicks | Filled with 0 | No | Before prediction |
| gsc_avg_position | Average ranking position | Filled with 0 | No | Before prediction |
| ga4_pageviews | Historical page traffic | Filled with 0 | No | Before prediction |
| ga4_sessions | Historical sessions | Filled with 0 | No | Before prediction |
| scroll_events | User engagement | Filled with 0 | No | Before prediction |
| impression_bucket | Engineered feature from impressions | N/A | Yes | Before prediction |

In [7]:
feature_vector.describe(include="all")

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_sessions,scroll_events,impression_bucket
count,413966,413966,413966,413966.000000,413966.000000,413966.000000,413966.000000,413966.000000,413966.000000,413966
unique,NaN,41,90489,NaN,NaN,NaN,NaN,NaN,NaN,3
top,NaN,client_23a62021009f63c4,content_2ac318b690824525,NaN,NaN,NaN,NaN,NaN,NaN,Low
freq,NaN,146493,31,NaN,NaN,NaN,NaN,NaN,NaN,233195
mean,2026-03-18 06:12:52.967828,NaN,NaN,206.144273,0.954677,12.604361,3.586903,3.139891,0.531693,NaN
min,2026-03-01 00:00:00,NaN,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN
25%,2026-03-12 00:00:00,NaN,NaN,16.000000,0.000000,3.268626,1.000000,1.000000,0.000000,NaN
50%,2026-03-18 00:00:00,NaN,NaN,76.000000,0.000000,7.629815,2.000000,1.000000,0.000000,NaN
75%,2026-03-25 00:00:00,NaN,NaN,217.000000,1.000000,19.480519,3.000000,3.000000,1.000000,NaN
max,2026-03-31 00:00:00,NaN,NaN,39305.000000,274.000000,377.000000,875.000000,792.000000,254.000000,NaN


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

# Leakage Hunt

The feature vector was reviewed for possible leakage.

Excluded:
- Future performance windows
- Label-derived columns
- Product decision flags

Only historical search and engagement information available before prediction is included.

In [8]:
safe_features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions",
    "scroll_events"
]

excluded_features = [
    "client_hash_id",
    "content_hash_id"
]

print("Safe Features:")
print(safe_features)

print("\nExcluded Features:")
print(excluded_features)

Safe Features:
['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'scroll_events']

Excluded Features:
['client_hash_id', 'content_hash_id']


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

# Excluded Features

| Field | Reason |
|-------|--------|
| client_hash_id | Identifier only; not predictive. |
| content_hash_id | Identifier only; not predictive. |
| Future outcome fields | Would introduce data leakage. |
| Product flags | Decision outputs, not model inputs. |

In [9]:
db.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT client_hash_id) AS unique_clients,
    COUNT(DISTINCT content_hash_id) AS unique_content
FROM march_data
""").df()

,total_rows,unique_clients,unique_content
0,9841378,55,331437


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.